In [1]:
!pip install pyspark

In [2]:
import pandas as pd

data = {
    "order_id": [101, 102, 103, 104],
    "supplier_id": [1, 2, 1, 3],
    "delivery_date": ["2026-05-10", "2026-05-18", "2026-05-12", "2026-05-20"],
    "delay_days": [4, -2, 2, -4],
    "is_delayed": [1, 0, 1, 0]
}

df = pd.DataFrame(data)

df.to_csv("processed_orders.csv", index=False)

print(df)

   order_id  supplier_id delivery_date  delay_days  is_delayed
0       101            1    2026-05-10           4           1
1       102            2    2026-05-18          -2           0
2       103            1    2026-05-12           2           1
3       104            3    2026-05-20          -4           0


In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SupplyChainProject") \
    .getOrCreate()

print("Spark Session Created")

Spark Session Created


In [4]:
spark_df = spark.read.csv(
    "processed_orders.csv",
    header=True,
    inferSchema=True
)

spark_df.show()

+--------+-----------+-------------+----------+----------+
|order_id|supplier_id|delivery_date|delay_days|is_delayed|
+--------+-----------+-------------+----------+----------+
|     101|          1|   2026-05-10|         4|         1|
|     102|          2|   2026-05-18|        -2|         0|
|     103|          1|   2026-05-12|         2|         1|
|     104|          3|   2026-05-20|        -4|         0|
+--------+-----------+-------------+----------+----------+



In [5]:
from pyspark.sql.functions import col

delayed_df = spark_df.filter(col("is_delayed") == 1)

delayed_df.show()

+--------+-----------+-------------+----------+----------+
|order_id|supplier_id|delivery_date|delay_days|is_delayed|
+--------+-----------+-------------+----------+----------+
|     101|          1|   2026-05-10|         4|         1|
|     103|          1|   2026-05-12|         2|         1|
+--------+-----------+-------------+----------+----------+



In [6]:
grouped_df = delayed_df.groupBy("supplier_id").count()

grouped_df.show()

+-----------+-----+
|supplier_id|count|
+-----------+-----+
|          1|    2|
+-----------+-----+



In [7]:
grouped_df.write.csv(
    "delayed_suppliers_output",
    header=True,
    mode="overwrite"
)

print("Output Saved")

Output Saved


In [8]:
from google.colab import files

files.download("processed_orders.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>